In [ ]:
import numpy as np
import pandas as pd
import os
import logging
from datetime import datetime
from pyproj import Proj
from scipy.interpolate import interp1d
from scipy.interpolate import PchipInterpolator
import math
import matplotlib.pyplot as plt
from pathlib import Path
from matplotlib.ticker import MultipleLocator


logger = logging.getLogger(__name__)

# -----------------------------------------------------------
# 전역(글로벌) Proj 객체: UTM zone 52N (EPSG:32652)
# 위경도 좌표를 UTM(동-북) 좌표로 변환하기 위해 사용
# -----------------------------------------------------------
_proj_utm52 = Proj("epsg:32652")


class DataProcessor:
    def __init__(self, window_size=150):
        self.window_size = window_size

    @staticmethod
    def load_and_preprocess_csv(
        file_path, skiprows=100, skipfooter=100, flag=False, zone=52, window_size=150
    ):
        if flag:
            df = pd.read_csv(
                file_path,
                skiprows=skiprows,
                skipfooter=skipfooter,
                na_values=["", "nan", "NaN"],
                engine="python",
            ).fillna(0)
        else:
            df = pd.read_csv(
                file_path, skiprows=skiprows, skipfooter=skipfooter, engine="python"
            )

        df.columns = [
            "Time",
            "Accelerometer x",
            "Accelerometer y",
            "Accelerometer z",
            "Gyroscope x",
            "Gyroscope y",
            "Gyroscope z",
            "Magnetometer x",
            "Magnetometer y",
            "Magnetometer z",
            "Orientation x",
            "Orientation y",
            "Orientation z",
            "Pressure",
            "Latitude",
            "Longitude",
            "Altitude",
            "Speed_GPS",
        ]

        df["Time"] = pd.to_datetime(df["Time"], format="%Y-%m-%d %H:%M:%S.%f")
        start_dt = df["Time"].iloc[0]
        df["Elapsed Time"] = (df["Time"] - start_dt).dt.total_seconds()

        df["Acc_Norm"] = np.linalg.norm(
            df[["Accelerometer x", "Accelerometer y", "Accelerometer z"]].values, axis=1
        )
        df["Gyro_Norm"] = np.linalg.norm(
            df[["Gyroscope x", "Gyroscope y", "Gyroscope z"]].values, axis=1
        )

        e, n, df = DataProcessor.llh_to_enu(df, flag, zone)
        v_10hz, dh_10Hz = DataProcessor.interpol_vAndh(e, n)
        X, Y = DataProcessor.makeXY(df, v_10hz, dh_10Hz, window_size=window_size)

        # y_e = []
        # y_n = []
        # dx = 0.0
        # dy = 0.0
        # heading = 0

        # stride = 5

        # plt.plot(e, n, ".-")
        # plt.axis("equal")
        # plt.grid()
        # plt.show()

        # for v, h in zip(Y[:, 0], Y[:, 1]):
        #     heading += h * (stride / window_size)
        #     dx += (v * (stride / window_size)) * np.cos(heading)
        #     dy += (v * (stride / window_size)) * np.sin(heading)
        #     y_e.append(dx)
        #     y_n.append(dy)

        # plt.plot(e, n, ".-", label="Y_true")
        # plt.plot(y_e, y_n, ".-", label="Y_pred")
        # plt.grid()
        # plt.axis("equal")
        # plt.show()

        # plt.plot(
        #     np.cumsum(np.degrees(Y[:, 1])) * (stride / window_size), ".-", label="Y_dh"
        # )
        # plt.gca().yaxis.set_major_locator(MultipleLocator(90))
        # plt.grid()
        # plt.legend()
        # plt.show()

        return df, X, Y

    @staticmethod
    def llh_to_enu(df, flag, zone=52):
        if flag:
            valid_gps_mask = (
                df["Latitude"].notna()
                & df["Longitude"].notna()
                & (df["Latitude"].astype(str).str.strip() != "")
                & (df["Longitude"].astype(str).str.strip() != "")
            )
            valid_lat = pd.to_numeric(
                df.loc[valid_gps_mask, "Latitude"], errors="coerce"
            )
            valid_lon = pd.to_numeric(
                df.loc[valid_gps_mask, "Longitude"], errors="coerce"
            )
            final_mask = valid_lat.notna() & valid_lon.notna()
            valid_lat = valid_lat[final_mask].values
            valid_lon = valid_lon[final_mask].values
            if len(valid_lat) == 0:
                raise ValueError("유효한 GPS 데이터가 없습니다.")
            proj_enu = Proj(proj="utm", zone=zone, ellps="WGS84", south=False)
            e0, n0 = proj_enu(valid_lon[0], valid_lat[0])
            e_valid, n_valid = proj_enu(valid_lon, valid_lat)
            e_valid -= e0
            n_valid -= n0
            df["E"], df["N"] = np.nan, np.nan
            df.loc[valid_gps_mask, "E"] = e_valid
            df.loc[valid_gps_mask, "N"] = n_valid
            e = df["E"][df["E"].notna()].values
            n = df["N"][df["N"].notna()].values

        else:
            df["Latitude"] = pd.to_numeric(df["Latitude"], errors="coerce")
            df["Longitude"] = pd.to_numeric(df["Longitude"], errors="coerce")
            proj_enu = Proj(proj="utm", zone=zone, ellps="WGS84", south=False)
            e_all, n_all = proj_enu(df["Longitude"].values, df["Latitude"].values)
            e0, n0 = proj_enu(df["Longitude"].iloc[0], df["Latitude"].iloc[0])
            df["E"] = e_all - e0
            df["N"] = n_all - n0
            M = len(df)
            n_sec = M // 50
            e, n = [], []
            for i in range(n_sec):
                idx = min(i * 50 + 25, M - 1)
                e.append(df["E"].iloc[idx])
                n.append(df["N"].iloc[idx])
            e = np.array(e)
            n = np.array(n)

        # -------------------------------------------------------
        # (3) Outlier 탐지
        # -------------------------------------------------------
        use_iqr = True
        dx = np.diff(e)
        dy = np.diff(n)
        speed = np.hypot(dx, dy)
        heading = np.arctan2(dy, dx)

        def remove_outliers_iqr(values, k=3.0):
            q1, q3 = np.percentile(values, [25, 75])
            iqr = q3 - q1
            return (values >= q1 - k * iqr) & (values <= q3 + k * iqr)

        def remove_outliers_zscore(values, threshold=3.0):
            mean, std = np.mean(values), np.std(values)
            return np.abs((values - mean) / (std + 1e-8)) < threshold

        if use_iqr:
            mask_speed = remove_outliers_iqr(speed)
            mask_heading = remove_outliers_iqr(np.degrees(heading))
        else:
            mask_speed = remove_outliers_zscore(speed)
            mask_heading = remove_outliers_zscore(np.degrees(heading))

        mask = mask_speed & mask_heading
        gps_idx_all = np.arange(1, len(e))
        bad_idx = gps_idx_all[~mask]

        print(f"총 GPS 샘플: {len(e)}, 이상치 GPS 샘플: {len(bad_idx)}")

        # -------------------------------------------------------
        # (4) 센서데이터 블록 drop (1Hz GPS → 50Hz 센서)
        # -------------------------------------------------------
        drop_idx = []
        for gi in bad_idx:
            start = gi * 50
            end = (gi + 1) * 50
            drop_idx.extend(range(start, min(end, len(df))))
        df = df.drop(drop_idx).reset_index(drop=True)

        if flag:
            # --- NaN 있는 경우: 유효 GPS만 모아서 다시 ENU trajectory 계산 ---
            valid_gps_mask = (
                df["Latitude"].notna()
                & df["Longitude"].notna()
                & (df["Latitude"].astype(str).str.strip() != "")
                & (df["Longitude"].astype(str).str.strip() != "")
            )
            valid_lat = pd.to_numeric(
                df.loc[valid_gps_mask, "Latitude"], errors="coerce"
            )
            valid_lon = pd.to_numeric(
                df.loc[valid_gps_mask, "Longitude"], errors="coerce"
            )
            final_mask = valid_lat.notna() & valid_lon.notna()
            valid_lat = valid_lat[final_mask].values
            valid_lon = valid_lon[final_mask].values

            if len(valid_lat) < 2:
                raise ValueError("유효 GPS가 drop 이후 2개 미만으로 남음")

            proj_enu = Proj(proj="utm", zone=zone, ellps="WGS84", south=False)
            e0, n0 = proj_enu(valid_lon[0], valid_lat[0])
            e_valid, n_valid = proj_enu(valid_lon, valid_lat)
            e, n = e_valid - e0, n_valid - n0

        else:
            # --- NaN 없는 경우: 센서 50Hz 중간 샘플 뽑기 ---
            M = len(df)
            n_sec = M // 50
            e, n = [], []
            for i in range(n_sec):
                idx = min(i * 50 + 25, M - 1)
                e.append(df["E"].iloc[idx])
                n.append(df["N"].iloc[idx])
            e, n = np.array(e), np.array(n)

        # # -------------------------------------------------------
        # # (5) 초기 heading 정렬 + 보간
        # # -------------------------------------------------------
        dx0, dy0 = e[1] - e[0], n[1] - n[0]
        theta0 = math.atan2(dy0, dx0)
        R0 = np.array(
            [
                [math.cos(-theta0), -math.sin(-theta0)],
                [math.sin(-theta0), math.cos(-theta0)],
            ]
        )
        coords = np.vstack([e - e[0], n - n[0]])
        rotated = R0 @ coords
        e_corr, n_corr = rotated[0], rotated[1]

        return e_corr, n_corr, df
    
    @staticmethod
    def fix_spikes_mean_neighbors(x, z_thresh=3.0, mean_dev_ratio=2.5, win=5):
        """
        x: 1D np.array
        z_thresh: 중앙값(Median)±MAD 기반 임계값 배수 (3.0 권장)
        mean_dev_ratio: 앞뒤 평균 대비 |x[i]-m| > mean_dev_ratio * (|x[i-1]-m|+|x[i+1]-m|)/2 형태의 추가 조건
                        (or 단순히 |x[i]-m| > mean_dev_ratio * local_std 로 바꿔도 됨)
        win: 롤링 중앙값/ MAD 윈도우(홀수 권장)

        반환: 보정된 x, 스파이크 마스크(원소가 True면 보정됨)
        """
        x = np.asarray(x, dtype=float).copy()
        n = len(x)
        if n < 3:
            return x, np.zeros(n, dtype=bool)

        # 롤링 중앙값과 MAD(중앙절대편차)
        s = pd.Series(x)
        med = s.rolling(win, center=True, min_periods=1).median().to_numpy()
        mad = s.rolling(win, center=True, min_periods=1).apply(
            lambda v: np.median(np.abs(v - np.median(v))), raw=True
        ).to_numpy()
        mad = np.where(mad < 1e-12, 1e-12, mad)  # 0 회피
        # z-score 유사 척도 (1.4826은 MAD→표준편차 보정 상수)
        z_like = np.abs(x - med) / (1.4826 * mad)

        # 이웃 평균 기준
        # m[i] = (x[i-1] + x[i+1]) / 2
        m = np.zeros_like(x)
        m[1:-1] = (x[:-2] + x[2:]) / 2
        m[0] = x[1]             # 맨 앞: 바로 다음 값으로
        m[-1] = x[-2]           # 맨 뒤: 바로 이전 값으로

        # 이웃 평균 대비 편차
        dev = np.abs(x - m)
        # 지역 스케일(간단히 앞뒤의 절댓편차 평균)
        local_scale = np.zeros_like(x)
        local_scale[1:-1] = (np.abs(x[1:-1] - x[:-2]) + np.abs(x[1:-1] - x[2:])) / 2
        local_scale = np.where(local_scale < 1e-6, 1e-6, local_scale)

        # 두 조건 중 하나라도 만족하면 스파이크로 간주
        spike_mask = (z_like > z_thresh) | (dev / local_scale > mean_dev_ratio)

        # 앞뒤 평균으로 대체 (엣지는 위에서 정의한 m 사용)
        x_fixed = x.copy()
        x_fixed[spike_mask] = m[spike_mask]

        return x_fixed, spike_mask

    @staticmethod
    def interpol_vAndh(e, n):
        delta_e = np.diff(e)
        delta_n = np.diff(n)

        v_1hz = np.hypot(delta_e, delta_n)
        v_1hz = (delta_e**2 + delta_n**2) ** 0.5
        v_1hz, mask_v = DataProcessor.fix_spikes_mean_neighbors(v_1hz, z_thresh=3.0, mean_dev_ratio=2.5, win=5)
        dh_1hz = np.diff(np.unwrap(np.arctan2(delta_n, delta_e)))

        # dh_1hz_clean_list = []

        # for dh in dh_1hz:
        #     if np.abs(dh) < np.radians(10):
        #         dh_1hz_clean_list.append(0)
        #     else:
        #         dh_1hz_clean_list.append(dh)

        origin_v = v_1hz[1:]
        origin_dh = dh_1hz

        N = len(origin_v)  # 1Hz 샘플 수
        t = np.arange(N, dtype=float)  # 0..N-1
        t_new = np.linspace(
            0.0, N - 1, (N - 1) * 10 + 1
        )  # ✅ 10Hz로 0~N-1초, 총 (N-1)*10+1개
        # t_new = np.linspace(0.0, N-1, (N-1)*50 + 1)  # ✅ 10Hz로 0~N-1초, 총 (N-1)*10+1개
        pv = PchipInterpolator(t, origin_v)
        pdh = PchipInterpolator(t, origin_dh)
        v_10Hz = pv(t_new)
        dh_10Hz = pdh(t_new)

        # plt.plot(v_10Hz, label="v_10Hz")
        # plt.grid()
        # plt.legend()
        # plt.show()

        # plt.plot(dh_10Hz, label="dh_10Hz")
        # plt.grid()
        # plt.legend()
        # plt.show()

        return v_10Hz, dh_10Hz

    @staticmethod
    def makeXY(df, v_10Hz, dh_10Hz, window_size):
        stride = 5
        sensor_cols = [
            "Accelerometer x",
            "Accelerometer y",
            "Accelerometer z",
            "Gyroscope x",
            "Gyroscope y",
            "Gyroscope z",
            "Acc_Norm",
            "Gyro_Norm",
        ]

        X = []
        for i in range(0, len(df) - window_size + 1, stride):
            window = (
                df[sensor_cols].iloc[i : i + window_size].values
            )  # (200, 8) numpy array
            X.append(window)

        X = np.array(X)  # (n, 200, 8)

        Y_v = []
        Y_dh = []

        offsets = [0, 10, 20]  # 1초 간격 (10Hz 기준)
        for i in range(len(dh_10Hz) - max(offsets)):
            # v: 1초 단위 4개를 합
            Y_v.append(
                v_10Hz[i + offsets[0]]
                + v_10Hz[i + offsets[1]]
                + v_10Hz[i + offsets[2]]
            )

            Y_dh.append(
                dh_10Hz[i + offsets[0]]
                + dh_10Hz[i + offsets[1]]
                + dh_10Hz[i + offsets[2]]
            )  
        Y = np.stack([Y_v, Y_dh], axis=1)
        X = X[: len(Y)]
        return X, Y

    @staticmethod
    def load_and_preprocess_csv_test(file_path, skiprows=50):
        # ... 기존 테스트용 전처리 로직 그대로 유지 ...
        df = pd.read_csv(file_path, skiprows=skiprows, skipfooter=100, engine="python")

        df.columns = [
            "Time",
            "Accelerometer x",
            "Accelerometer y",
            "Accelerometer z",
            "Gyroscope x",
            "Gyroscope y",
            "Gyroscope z",
            "Magnetometer x",
            "Magnetometer y",
            "Magnetometer z",
            "Orientation x",
            "Orientation y",
            "Orientation z",
            "Pressure",
            "Latitude",
            "Longitude",
            "Altitude",
            "Speed_GPS",
        ]
        df["Time"] = pd.to_datetime(df["Time"], format="%Y-%m-%d %H:%M:%S.%f")
        start_dt = df["Time"].iloc[0]
        df["Elapsed Time"] = (df["Time"] - start_dt).dt.total_seconds()

        df["Acc_Norm"] = np.linalg.norm(
            df[["Accelerometer x", "Accelerometer y", "Accelerometer z"]].values, axis=1
        )
        df["Gyro_Norm"] = np.linalg.norm(
            df[["Gyroscope x", "Gyroscope y", "Gyroscope z"]].values, axis=1
        )

        return df

In [ ]:
config = {
    "looking_left01.csv": {"skiprows": 300, "flag": False, "zone": 52},
    "looking_left02.csv": {"skiprows": 150, "flag": False, "zone": 52},
    "looking_left03.csv": {"skiprows": 250, "flag": False, "zone": 52},
    "looking_left04.csv": {"skiprows": 150, "flag": False, "zone": 52},
    "looking_left05.csv": {"skiprows": 300, "flag": True, "zone": 52},
    
    "looking_right01.csv": {"skiprows": 200, "flag": False, "zone": 52},
    "looking_right02.csv": {"skiprows": 100, "flag": False, "zone": 52},
    "looking_right03.csv": {"skiprows": 200, "flag": False, "zone": 52},
    "looking_right04.csv": {"skiprows": 350, "flag": True, "zone": 52},
    
    "looking_lr01.csv": {"skiprows": 100, "flag": False, "zone": 52},
    "looking_lr02.csv": {"skiprows": 500, "flag": True, "zone": 52},
    
    "swing_left01.csv": {"skiprows": 200, "flag": False, "zone": 52},
    "swing_left02.csv": {"skiprows": 400, "flag": False, "zone": 52},
    "swing_left03.csv": {"skiprows": 300, "flag": False, "zone": 52},
    "swing_left04.csv": {"skiprows": 300, "flag": True, "zone": 52},
    "swing_left05.csv": {"skiprows": 200, "flag": True, "zone": 52},
    "swing_left06.csv": {"skiprows": 200, "flag": True, "zone": 52},
    "swing_left07.csv": {"skiprows": 200, "flag": True, "zone": 52},
    "swing_left08.csv": {"skiprows": 300, "flag": True, "zone": 52},
    "swing_left09.csv": {"skiprows": 300, "flag": True, "zone": 52},
    "swing_left10.csv": {"skiprows": 300, "flag": True, "zone": 52},
    "swing_left11.csv": {"skiprows": 300, "flag": True, "zone": 52},
    "swing_left12.csv": {"skiprows": 300, "flag": True, "zone": 52},

    "swing_right01.csv": {"skiprows": 150, "flag": False, "zone": 52},
    "swing_right02.csv": {"skiprows": 300, "flag": False, "zone": 52},
    "swing_right03.csv": {"skiprows": 300, "flag": False, "zone": 52},
    "swing_right04.csv": {"skiprows": 300, "flag": True, "zone": 52},
    "swing_right05.csv": {"skiprows": 200, "flag": True, "zone": 52},
    "swing_right06.csv": {"skiprows": 200, "flag": True, "zone": 52},
    "swing_right07.csv": {"skiprows": 300, "flag": True, "zone": 52},
    "swing_right08.csv": {"skiprows": 500, "flag": True, "zone": 52},
    "swing_right09.csv": {"skiprows": 300, "flag": True, "zone": 52},
    "swing_right10.csv": {"skiprows": 300, "flag": True, "zone": 52},
    "swing_right11.csv": {"skiprows": 300, "flag": True, "zone": 52},
    "swing_right12.csv": {"skiprows": 300, "flag": True, "zone": 52},
    
    "calling_left01.csv": {"skiprows": 200, "flag": True, "zone": 52}, 
    "calling_left02.csv": {"skiprows": 200, "flag": True, "zone": 52}, 
    "calling_left03.csv": {"skiprows": 200, "flag": True, "zone": 52}, 
    "calling_left04.csv": {"skiprows": 200, "flag": True, "zone": 52}, 
    "calling_left05.csv": {"skiprows": 200, "flag": True, "zone": 52}, 
    "calling_left06.csv": {"skiprows": 200, "flag": True, "zone": 52}, 
    
    "calling_right01.csv": {"skiprows": 200, "flag": True, "zone": 52},
    "calling_right02.csv": {"skiprows": 200, "flag": True, "zone": 52},
    "calling_right03.csv": {"skiprows": 200, "flag": True, "zone": 52},
    "calling_right04.csv": {"skiprows": 200, "flag": True, "zone": 52},
    "calling_right05.csv": {"skiprows": 200, "flag": True, "zone": 52},
    "calling_right06.csv": {"skiprows": 200, "flag": True, "zone": 52},

}

default_config = {"skiprows": 100, "flag": False, "zone": 52}

# =========================
# 전역 설정
# =========================
BASE_DIR = os.getcwd()
FS = 50  # Hz

# 학습 데이터 경로들
swing_learnData_path = [
    os.path.join(BASE_DIR, "data", "learn_data", "swing", "swing_left01.csv"),
    os.path.join(BASE_DIR, "data", "learn_data", "swing", "swing_left02.csv"),
    os.path.join(BASE_DIR, "data", "learn_data", "swing", "swing_left03.csv"),
    os.path.join(BASE_DIR, "data", "learn_data", "swing", "swing_left04.csv"),
    os.path.join(BASE_DIR, "data", "learn_data", "swing", "swing_left05.csv"),
    os.path.join(BASE_DIR, "data", "learn_data", "swing", "swing_left06.csv"),
    os.path.join(BASE_DIR, "data", "learn_data", "swing", "swing_left07.csv"),
    # os.path.join(BASE_DIR, "data", "learn_data", "swing", "swing_left08.csv"),
    # os.path.join(BASE_DIR, "data", "learn_data", "swing", "swing_left09.csv"),
    # os.path.join(BASE_DIR, "data", "learn_data", "swing", "swing_left10.csv"),
    # os.path.join(BASE_DIR, "data", "learn_data", "swing", "swing_left11.csv"),
    # os.path.join(BASE_DIR, "data", "learn_data", "swing", "swing_left12.csv"),
    
    os.path.join(BASE_DIR, "data", "learn_data", "swing", "swing_right01.csv"),
    os.path.join(BASE_DIR, "data", "learn_data", "swing", "swing_right02.csv"),
    os.path.join(BASE_DIR, "data", "learn_data", "swing", "swing_right03.csv"),
    os.path.join(BASE_DIR, "data", "learn_data", "swing", "swing_right04.csv"),
    os.path.join(BASE_DIR, "data", "learn_data", "swing", "swing_right05.csv"),
    os.path.join(BASE_DIR, "data", "learn_data", "swing", "swing_right06.csv"),
    os.path.join(BASE_DIR, "data", "learn_data", "swing", "swing_right07.csv"),
    # os.path.join(BASE_DIR, "data", "learn_data", "swing", "swing_right08.csv"),
    # os.path.join(BASE_DIR, "data", "learn_data", "swing", "swing_right09.csv"),
    # os.path.join(BASE_DIR, "data", "learn_data", "swing", "swing_right10.csv"),
    # os.path.join(BASE_DIR, "data", "learn_data", "swing", "swing_right11.csv"),
    # os.path.join(BASE_DIR, "data", "learn_data", "swing", "swing_right12.csv"),
    
    # os.path.join(BASE_DIR, "data", "learn_data", "calling", "calling_left01.csv"),
    # os.path.join(BASE_DIR, "data", "learn_data", "calling", "calling_left02.csv"),
    # os.path.join(BASE_DIR, "data", "learn_data", "calling", "calling_left03.csv"),
    # os.path.join(BASE_DIR, "data", "learn_data", "calling", "calling_left04.csv"),
    # os.path.join(BASE_DIR, "data", "learn_data", "calling", "calling_left05.csv"),
    # os.path.join(BASE_DIR, "data", "learn_data", "calling", "calling_left06.csv"),
    
    # os.path.join(BASE_DIR, "data", "learn_data", "calling", "calling_right01.csv"),
    # os.path.join(BASE_DIR, "data", "learn_data", "calling", "calling_right02.csv"),
    # os.path.join(BASE_DIR, "data", "learn_data", "calling", "calling_right03.csv"),
    # os.path.join(BASE_DIR, "data", "learn_data", "calling", "calling_right04.csv"),
    # os.path.join(BASE_DIR, "data", "learn_data", "calling", "calling_right05.csv"),
    # os.path.join(BASE_DIR, "data", "learn_data", "calling", "calling_right06.csv"),
]

looking_learnData_path = [
    # os.path.join(BASE_DIR, "data", "learn_data", "looking", "looking_left01.csv"),
    # os.path.join(BASE_DIR, "data", "learn_data", "looking", "looking_left02.csv"),
    # os.path.join(BASE_DIR, "data", "learn_data", "looking", "looking_left03.csv"),
    # os.path.join(BASE_DIR, "data", "learn_data", "looking", "looking_left04.csv"),
    # os.path.join(BASE_DIR, "data", "learn_data", "looking", "looking_left05.csv"),
    # os.path.join(BASE_DIR, "data", "learn_data", "looking", "looking_right01.csv"),
    # os.path.join(BASE_DIR, "data", "learn_data", "looking", "looking_right02.csv"),
    # os.path.join(BASE_DIR, "data", "learn_data", "looking", "looking_right03.csv"),
    # os.path.join(BASE_DIR, "data", "learn_data", "looking", "looking_right04.csv"),
    
    os.path.join(BASE_DIR, "data", "learn_data", "looking", "looking_lr01.csv"),
    os.path.join(BASE_DIR, "data", "learn_data", "looking", "looking_lr02.csv"),
]


# =========================
# 그룹 로딩 & 길이 합산
# =========================
def load_group(paths, loader_fn, cfg=None, default_cfg=None, fs=FS):
    """
    한 그룹(paths) 로드/전처리, 파일별 옵션 자동 적용.
      - loader_fn(path, **opts) -> (df, X, Y)
      - cfg: 파일명별 옵션 dict
      - default_cfg: 기본 옵션 dict
    """
    cfg = cfg or {}
    default_cfg = default_cfg or {}
    df_list, X_list, Y_list = [], [], []
    total_min = 0.0

    for idx, p in enumerate(paths, start=1):
        fname = Path(p).name
        opts = {**default_cfg, **cfg.get(fname, {})}  # default + per-file override

        try:
            df_temp, X_temp, Y_temp = loader_fn(p, **opts)
        except Exception as e:
            print(f"[에러] {fname}: {e}")
            continue

        df_list.append(df_temp)
        X_list.append(X_temp)
        Y_list.append(Y_temp)

        minutes = len(df_temp) / fs / 60.0
        print(
            f"[{idx:02d}] {fname:<32} 길이:{len(df_temp):7d}  ≈ {minutes:6.2f} 분  opts={opts}"
        )
        total_min += minutes

    print(f"--> 그룹 합계: {total_min:.2f} 분\n")
    return df_list, X_list, Y_list, total_min


# =========================
# 좌/우/혼합 분량 요약
# =========================
def summarize_turn_minutes(df_list, paths, fs=FS):
    """
    df_list: 각 파일의 DataFrame 리스트
    paths  : 각 파일의 경로 리스트 (df_list와 동일 순서)
    fs     : 샘플링 주파수(Hz)
    return : (per_file_rows, totals)
    """
    rows = []
    tot_left = 0.0
    tot_right = 0.0
    for df, p in zip(df_list, paths):
        fname = Path(p).name
        name = fname.lower()
        minutes = len(df) / fs / 60.0

        if ("lr" in name) or ("left" in name and "right" in name):
            # 혼합 데이터 → 좌/우 반반
            left_min = minutes / 2.0
            right_min = minutes / 2.0
            tag = "lr(50/50)"
        elif "left" in name:
            left_min = minutes
            right_min = 0.0
            tag = "left"
        elif "right" in name:
            left_min = 0.0
            right_min = minutes
            tag = "right"
        else:
            left_min = 0.0
            right_min = 0.0
            tag = "unknown"

        tot_left += left_min
        tot_right += right_min
        rows.append(
            {
                "file": fname,
                "minutes": round(minutes, 2),
                "tag": tag,
                "left_min": round(left_min, 2),
                "right_min": round(right_min, 2),
            }
        )

    totals = {
        "left": round(tot_left, 2),
        "right": round(tot_right, 2),
        "total": round(tot_left + tot_right, 2),
    }
    return rows, totals


def pretty_print_summary(title, rows, totals):
    print(f"\n=== {title} ===")
    print(f"{'file':32} {'min':>7}  {'tag':10} {'left_min':>9} {'right_min':>10}")
    for r in rows:
        print(
            f"{r['file']:<32} {r['minutes']:7.2f}  {r['tag']:<10} {r['left_min']:9.2f} {r['right_min']:10.2f}"
        )
    print(
        f"-- 합계: left={totals['left']:.2f} min | right={totals['right']:.2f} min | total={totals['total']:.2f} min"
    )


# (선택) 판다스 표/CSV 저장
def to_dataframe(rows):
    return pd.DataFrame(
        rows, columns=["file", "minutes", "tag", "left_min", "right_min"]
    )


def save_summary_csv(prefix, rows, totals, out_dir="outputs"):
    os.makedirs(out_dir, exist_ok=True)
    df = to_dataframe(rows)
    df.to_csv(
        os.path.join(out_dir, f"{prefix}_per_file.csv"),
        index=False,
        encoding="utf-8-sig",
    )
    pd.DataFrame([totals]).to_csv(
        os.path.join(out_dir, f"{prefix}_totals.csv"), index=False, encoding="utf-8-sig"
    )


# =========================
# 실행부
# =========================
if __name__ == "__main__":
    loader1 = lambda path, **opts: DataProcessor.load_and_preprocess_csv(path, **opts)

    print("=== SWING 데이터 ===")
    swing_df_list, swing_X_list, swing_Y_list, swing_min = load_group(
        swing_learnData_path, loader_fn=loader1, cfg=config, default_cfg=default_config
    )

    print("=== LOOKING 데이터 ===")
    looking_df_list, looking_X_list, looking_Y_list, looking_min = load_group(
        looking_learnData_path,
        loader_fn=loader1,
        cfg=config,
        default_cfg=default_config,
    )

    print(
        f"요약) SWING: {swing_min:.2f} 분 | LOOKING: {looking_min:.2f} 분 | 총합: {swing_min + looking_min:.2f} 분"
    )

    # 좌/우/혼합 분량 요약
    swing_rows, swing_tot = summarize_turn_minutes(
        swing_df_list, swing_learnData_path, fs=FS
    )
    pretty_print_summary("SWING", swing_rows, swing_tot)

    looking_rows, looking_tot = summarize_turn_minutes(
        looking_df_list, looking_learnData_path, fs=FS
    )
    pretty_print_summary("LOOKING", looking_rows, looking_tot)

    # 전체 합산
    all_rows = swing_rows + looking_rows
    all_tot = {
        "left": round(swing_tot["left"] + looking_tot["left"], 2),
        "right": round(swing_tot["right"] + looking_tot["right"], 2),
        "total": round(swing_tot["total"] + looking_tot["total"], 2),
    }
    pretty_print_summary("전체(SWING + LOOKING)", all_rows, all_tot)